In [1]:
import deeptools as dt
import pandas as pd

In [2]:
# func

def split_and_exp(csv):
    bedpe = pd.read_csv(csv, sep="\t", header=None)
    left_anchors = bedpe.iloc[:, [0, 1, 2]].set_axis([0, 1, 2], axis=1)
    left_anchors["anchor"] = "left"
    right_anchors = bedpe.iloc[:, [3, 4, 5]].set_axis([0, 1, 2], axis=1)
    right_anchors["anchor"] = "right"
    all_anchors_all = pd.concat([left_anchors, right_anchors], ignore_index=True)
    return all_anchors_all

In [3]:
# ctcf

In [4]:
# first split bedpe to left and right anchors and then concatenate them into a single bed file
ctcf_bedpe = pd.read_csv("/usr/users/papantonis1/aman/microc_project/loop_calling_premade_hic/merged_loops_3tools/ctcf_associated_loops_eitheranchoroverlap_ctrl.bedpe", sep="\t", header=None)
ctcf_bedpe

,0,1,2,3,4,5
0,chr1,8022500,8027500,chr1,8312500,8317500
1,chr1,8025000,8030000,chr1,8090000,8095000
2,chr1,8025000,8030000,chr1,8170000,8175000
3,chr1,8025000,8030000,chr1,8240000,8245000
4,chr1,8095000,8100000,chr1,8310000,8315000
...,...,...,...,...,...,...
3355,chr9,114805000,114810000,chr9,114905000,114910000
3356,chr9,129882500,129887500,chr9,130102500,130107500
3357,chr9,135082500,135087500,chr9,135342500,135347500
3358,chrX,68822500,68827500,chrX,69382500,69387500


In [5]:
#extracting cols
left_anchors = ctcf_bedpe.iloc[:, [0, 1, 2]].set_axis([0, 1, 2], axis=1)
left_anchors["anchor"] = "left"
#left_anchors
right_anchors = ctcf_bedpe.iloc[:, [3, 4, 5]].set_axis([0, 1, 2], axis=1)
right_anchors["anchor"] = "right"
right_anchors

,0,1,2,anchor
0,chr1,8312500,8317500,right
1,chr1,8090000,8095000,right
2,chr1,8170000,8175000,right
3,chr1,8240000,8245000,right
4,chr1,8310000,8315000,right
...,...,...,...,...
3355,chr9,114905000,114910000,right
3356,chr9,130102500,130107500,right
3357,chr9,135342500,135347500,right
3358,chrX,69382500,69387500,right


In [6]:
#concat
all_anchors = pd.concat([left_anchors, right_anchors], ignore_index=True)
all_anchors

,0,1,2,anchor
0,chr1,8022500,8027500,left
1,chr1,8025000,8030000,left
2,chr1,8025000,8030000,left
3,chr1,8025000,8030000,left
4,chr1,8095000,8100000,left
...,...,...,...,...
6715,chr9,114905000,114910000,right
6716,chr9,130102500,130107500,right
6717,chr9,135342500,135347500,right
6718,chrX,69382500,69387500,right


In [7]:
all_anchors.to_csv("ctrl_ctcf_anchors_eitheranchoroverlap_derived.csv", sep="\t",header=False,index=False)

In [8]:
ctcf_rbp1 = split_and_exp("/usr/users/papantonis1/aman/microc_project/loop_calling_premade_hic/merged_loops_3tools/ctcf_associated_loops_eitheranchoroverlap_rbp1.bedpe")
ctcf_rbp1.to_csv("rbp1_ctcf_anchors_eitheranchoroverlap_derived.csv", sep="\t",header=False,index=False)

In [9]:
# h3k27me3

In [10]:
#ctrl
h3k27_ctrl = split_and_exp("/usr/users/papantonis1/aman/microc_project/loop_calling_premade_hic/merged_loops_3tools/h3k27me3_associated_loops_eitheranchoroverlap_ctrl.bedpe")

In [11]:
h3k27_ctrl.to_csv("ctrl_h3k27_anchors_eitheranchoroverlap_derived.csv", sep="\t",header=False,index=False)

In [12]:
#rbp1
h3k27_rbp1 = split_and_exp("/usr/users/papantonis1/aman/microc_project/loop_calling_premade_hic/merged_loops_3tools/h3k27me3_associated_loops_eitheranchoroverlap_rbp1.bedpe")
h3k27_rbp1.to_csv("rbp1_h3k27_anchors_eitheranchoroverlap_derived.csv", sep="\t",header=False,index=False)

In [14]:
## using subprocess
import subprocess

result = subprocess.run([
    "computeMatrix", "reference-point",
    "--referencePoint", "center",
    "-b", "2500", "-a", "2500",
    "-R",
    "/usr/users/papantonis1/aman/microc_project/loop_calling_premade_hic/jupyter_notes/ctrl_ctcf_anchors_eitheranchoroverlap_derived.csv",
    "/usr/users/papantonis1/aman/microc_project/loop_calling_premade_hic/jupyter_notes/ctrl_h3k27_anchors_eitheranchoroverlap_derived.csv",
    "-S",
    "/usr/users/papantonis1/aman/microc_data/nadine_macro/C_CTCF.bw_RPGC.bw",
    "/usr/users/papantonis1/aman/microc_data/cut_n_tag/nadine_cut_tag/nadine_cut_tag/C_H3K27me3_results/Aligned_files/Bigwig_scaled/C_H3K27me3.bw_RPGC.bw",
    "--skipZeros",
    "--missingDataAsZero",
    "-out", "chromatin_signal_matrix.gz",
    "--outFileSortedRegions", "anchors_sorted.bed"
],
capture_output=True,
text=True
)

print(result.stdout)

print("STDERR:")
print(result.stderr)

# Optional: check if it failed
if result.returncode != 0:
    print("Error: computeMatrix failed")



STDERR:
Skipping chr10:20852500-20857500, due to being absent in the computeMatrix output.
Skipping chr10:32262500-32267500, due to being absent in the computeMatrix output.
Skipping chr10:69005000-69010000, due to being absent in the computeMatrix output.
Skipping chr10:113132500-113137500, due to being absent in the computeMatrix output.
Skipping chr11:10202500-10207500, due to being absent in the computeMatrix output.
Skipping chr12:62802500-62807500, due to being absent in the computeMatrix output.
Skipping chr13:80032500-80037500, due to being absent in the computeMatrix output.
Skipping chr16:55555000-55560000, due to being absent in the computeMatrix output.
Skipping chr17:15840000-15845000, due to being absent in the computeMatrix output.
Skipping chr2:47852500-47857500, due to being absent in the computeMatrix output.
Skipping chr21:27710000-27715000, due to being absent in the computeMatrix output.
Skipping chr3:45095000-45100000, due to being absent in the computeMatrix out

In [2]:
import subprocess

subprocess.run([
    "plotHeatmap",
    "-m", "chromatin_signal_matrix.gz",
    "-out", "chromatin_signal_heatmap_labeled.png",
    "--colorMap", "RdBu_r",
    "--whatToShow", "heatmap and colorbar",
    "--zMin", "0",
    "--zMax", "200",
    "--refPointLabel", "Anchor center",
    "--regionsLabel", "CTCF_anchors", "H3K27me3_anchors",
    "--samplesLabel", "CTCF_signal", "H3K27me3_signal",
    "--plotTitle", "CUT&TAG Signal Heatmap"
])


CompletedProcess(args=['plotHeatmap', '-m', 'chromatin_signal_matrix.gz', '-out', 'chromatin_signal_heatmap_labeled.png', '--colorMap', 'RdBu_r', '--whatToShow', 'heatmap and colorbar', '--zMin', '0', '--zMax', '200', '--refPointLabel', 'Anchor center', '--regionsLabel', 'CTCF_anchors', 'H3K27me3_anchors', '--samplesLabel', 'CTCF_signal', 'H3K27me3_signal', '--plotTitle', 'CUT&TAG Signal Heatmap'], returncode=0)

---

In [16]:
# doing it individually 
# h3k27me3

## using subprocess
import subprocess

result = subprocess.run([
    "computeMatrix", "reference-point",
    "--referencePoint", "center",
    "-b", "2500", "-a", "2500",
    "-R",
    "/usr/users/papantonis1/aman/microc_project/loop_calling_premade_hic/jupyter_notes/ctrl_h3k27_anchors_eitheranchoroverlap_derived.csv",
    "-S",
    "/usr/users/papantonis1/aman/microc_data/cut_n_tag/nadine_cut_tag/nadine_cut_tag/C_H3K27me3_results/Aligned_files/Bigwig_scaled/C_H3K27me3.bw_RPGC.bw",
    "--skipZeros",
    "--missingDataAsZero",
    "-out", "ctrl_h3k27_chromatin_signal_matrix.gz",
    "--outFileSortedRegions", "ctrl_h3k27_anchors_sorted.bed"
],
capture_output=True,
text=True
)

print(result.stdout)

print("STDERR:")
print(result.stderr)

# Optional: check if it failed
if result.returncode != 0:
    print("Error: computeMatrix failed")



STDERR:
Skipping chr1:100922500-100927500, due to being absent in the computeMatrix output.
Skipping chr10:101055000-101060000, due to being absent in the computeMatrix output.
Skipping chr10:119750000-119755000, due to being absent in the computeMatrix output.
Skipping chr11:20392500-20397500, due to being absent in the computeMatrix output.
Skipping chr11:82905000-82910000, due to being absent in the computeMatrix output.
Skipping chr11:113805000-113810000, due to being absent in the computeMatrix output.
Skipping chr12:66142500-66147500, due to being absent in the computeMatrix output.
Skipping chr14:61450000-61455000, due to being absent in the computeMatrix output.
Skipping chr14:74102500-74107500, due to being absent in the computeMatrix output.
Skipping chr14:99502500-99507500, due to being absent in the computeMatrix output.
Skipping chr16:71225000-71230000, due to being absent in the computeMatrix output.
Skipping chr17:47672500-47677500, due to being absent in the computeMat

In [20]:
import subprocess

subprocess.run([
    "plotHeatmap",
    "-m", "ctrl_h3k27_chromatin_signal_matrix.gz",
    "-out", "ctrl_h3k27_chromatin_signal_heatmap_labeled.png",
    "--colorMap", "RdBu_r",
    "--whatToShow", "heatmap and colorbar",
    "--zMin", "0",
    "--zMax", "200",
    "--refPointLabel", "Anchor center",
    "--regionsLabel", "H3K27me3_anchors",
    "--samplesLabel", "H3K27me3_signal",
    "--plotTitle", "CUT&TAG Signal Heatmap"
])


CompletedProcess(args=['plotHeatmap', '-m', 'ctrl_h3k27_chromatin_signal_matrix.gz', '-out', 'ctrl_h3k27_chromatin_signal_heatmap_labeled.png', '--colorMap', 'RdBu_r', '--whatToShow', 'heatmap and colorbar', '--zMin', '0', '--zMax', '200', '--refPointLabel', 'Anchor center', '--regionsLabel', 'H3K27me3_anchors', '--samplesLabel', 'H3K27me3_signal', '--plotTitle', 'CUT&TAG Signal Heatmap'], returncode=0)

---

In [18]:
# doing it individually 
# ctcf

## using subprocess
import subprocess

result = subprocess.run([
    "computeMatrix", "reference-point",
    "--referencePoint", "center",
    "-b", "2500", "-a", "2500",
    "-R",
    "/usr/users/papantonis1/aman/microc_project/loop_calling_premade_hic/jupyter_notes/ctrl_ctcf_anchors_eitheranchoroverlap_derived.csv",
    "-S",
    "/usr/users/papantonis1/aman/microc_data/nadine_macro/C_CTCF.bw_RPGC.bw",
    "--skipZeros",
    "--missingDataAsZero",
    "-out", "ctrl_ctcf_chromatin_signal_matrix.gz",
    "--outFileSortedRegions", "ctrl_ctcf_anchors_sorted.bed"
],
capture_output=True,
text=True
)

print(result.stdout)

print("STDERR:")
print(result.stderr)

# Optional: check if it failed
if result.returncode != 0:
    print("Error: computeMatrix failed")




STDERR:
Skipping chr1:13972500-13977500, due to being absent in the computeMatrix output.
Skipping chr1:33210000-33215000, due to being absent in the computeMatrix output.
Skipping chr1:44382500-44387500, due to being absent in the computeMatrix output.
Skipping chr1:49202500-49207500, due to being absent in the computeMatrix output.
Skipping chr1:53985000-53990000, due to being absent in the computeMatrix output.
Skipping chr1:83912500-83917500, due to being absent in the computeMatrix output.
Skipping chr1:84132500-84137500, due to being absent in the computeMatrix output.
Skipping chr1:172432500-172437500, due to being absent in the computeMatrix output.
Skipping chr1:183642500-183647500, due to being absent in the computeMatrix output.
Skipping chr1:220202500-220207500, due to being absent in the computeMatrix output.
Skipping chr1:225550000-225555000, due to being absent in the computeMatrix output.
Skipping chr10:20852500-20857500, due to being absent in the computeMatrix output

In [21]:
import subprocess

subprocess.run([
    "plotHeatmap",
    "-m", "ctrl_ctcf_chromatin_signal_matrix.gz",
    "-out", "ctrl_ctcf_chromatin_signal_heatmap_labeled.png",
    "--colorMap", "RdBu_r",
    "--whatToShow", "heatmap and colorbar",
    "--zMin", "0",
    "--zMax", "200",
    "--refPointLabel", "Anchor center",
    "--regionsLabel", "CTCF_anchors",
    "--samplesLabel", "CTCF_signal",
    "--plotTitle", "CUT&TAG Signal Heatmap"
])

CompletedProcess(args=['plotHeatmap', '-m', 'ctrl_ctcf_chromatin_signal_matrix.gz', '-out', 'ctrl_ctcf_chromatin_signal_heatmap_labeled.png', '--colorMap', 'RdBu_r', '--whatToShow', 'heatmap and colorbar', '--zMin', '0', '--zMax', '200', '--refPointLabel', 'Anchor center', '--regionsLabel', 'CTCF_anchors', '--samplesLabel', 'CTCF_signal', '--plotTitle', 'CUT&TAG Signal Heatmap'], returncode=0)

In [7]:
# doing it individually 
# H3K27me3 - TSS upregulated

## using subprocess
import subprocess

result = subprocess.run([
    "computeMatrix", "reference-point",
    "--referencePoint", "center",
    "-b", "2500", "-a", "2500",
    "-R", "/usr/users/papantonis1/aman/rnaseq_data/upregulated_transcript_level_TSS.bed",
    "-S", "/usr/users/papantonis1/aman/microc_data/cut_n_tag/nadine_cut_tag/nadine_cut_tag/C_H3K27me3_results/Aligned_files/Bigwig_scaled/CPI_H3K27me3.bw_RPGC.bw",
    "--skipZeros",
    "--missingDataAsZero",
    "-out", "ctrl_H3K27me3_TSS_matrix.gz",
    "--outFileSortedRegions", "ctrl_H3K27me3_TSS_sorted.bed"
], capture_output=True, text=True)

print(result.stdout)
print("STDERR:", result.stderr)
if result.returncode != 0:
    print("Error: computeMatrix failed")





STDERR: Skipping ENST00000544305.5, due to being absent in the computeMatrix output.
Skipping ENST00000374630.8, due to being absent in the computeMatrix output.
Skipping ENST00000400191.7, due to being absent in the computeMatrix output.
Skipping ENST00000374632.7, due to being absent in the computeMatrix output.
Skipping ENST00000633167.1, due to being absent in the computeMatrix output.
Skipping ENST00000373836.4, due to being absent in the computeMatrix output.
Skipping ENST00000632964.1, due to being absent in the computeMatrix output.
Skipping ENST00000310955.11, due to being absent in the computeMatrix output.
Skipping ENST00000372462.1, due to being absent in the computeMatrix output.
Skipping ENST00000478882.1, due to being absent in the computeMatrix output.
Skipping ENST00000641455.1, due to being absent in the computeMatrix output.
Skipping ENST00000462520.5, due to being absent in the computeMatrix output.
Skipping ENST00000368918.8, due to being absent in the computeMatr

In [8]:
subprocess.run([
    "plotHeatmap",
    "-m", "ctrl_H3K27me3_TSS_matrix.gz",
    "-out", "ctrl_H3K27me3_TSS_heatmap.png",
    "--colorMap", "RdBu_r",
    "--whatToShow", "heatmap and colorbar",
    "--zMin", "0",
    "--zMax", "200",
    "--refPointLabel", "TSS",
    "--regionsLabel", "Upregulated_TSS",
    "--samplesLabel", "H3K27me3_signal",
    "--plotTitle", "H3K27me3 Signal at Upregulated TSSs (CTRL)"
])

CompletedProcess(args=['plotHeatmap', '-m', 'ctrl_H3K27me3_TSS_matrix.gz', '-out', 'ctrl_H3K27me3_TSS_heatmap.png', '--colorMap', 'RdBu_r', '--whatToShow', 'heatmap and colorbar', '--zMin', '0', '--zMax', '200', '--refPointLabel', 'TSS', '--regionsLabel', 'Upregulated_TSS', '--samplesLabel', 'H3K27me3_signal', '--plotTitle', 'H3K27me3 Signal at Upregulated TSSs (CTRL)'], returncode=0)